In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [21]:
!pip install openai -q

In [22]:
import json, os, re, time, subprocess, ast
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

INPUT_DIR = '/kaggle/input/datasets/apr-dataset'
OUTPUT_DIR = '/kaggle/working'

KB_PATH = f'{INPUT_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
OUTPUT_PATH = f'{OUTPUT_DIR}/phase2_batch_results_gpt54mini_k10.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 10
GPT_MODEL = "gpt-5.4-mini"  # verify exact string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded KB: {len(pattern_knowledge_base)} patterns')

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))

Loaded KB: 6 patterns
Loaded 302 test samples
Test cases dir exists: True


In [31]:
def safe_parse(code):
    try:
        return ast.parse(code)
    except SyntaxError:
        return None

def has_append_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'append':
                    return True
    return False

def has_add_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'add':
                    return True
    return False

def has_membership_or_index_in_loop(tree):
    for node in ast.walk(tree):
        if isinstance(node, (ast.For, ast.While)):
            for sub in ast.walk(node):
                if isinstance(sub, ast.Compare) and any(isinstance(op, ast.In) for op in sub.ops):
                    return True
                if isinstance(sub, ast.Call) and isinstance(sub.func, ast.Attribute) and sub.func.attr == 'index':
                    return True
    return False

def max_nesting_depth(tree):
    max_d = [0]
    def _walk(node, depth):
        if isinstance(node, (ast.For, ast.While)):
            depth += 1
            max_d[0] = max(max_d[0], depth)
        for child in ast.iter_child_nodes(node):
            _walk(child, depth)
    _walk(tree, 0)
    return max_d[0]

def count_same_level_loops(tree):
    count = [0]
    def _walk(node, inside_loop):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.For, ast.While)):
                if not inside_loop:
                    count[0] += 1
                _walk(child, inside_loop=True)
            else:
                _walk(child, inside_loop=inside_loop)
    _walk(tree, False)
    return count[0]

def extract_structural_features(code):
    tree = safe_parse(code)
    if tree is None:
        return None
    return {
        "has_append_in_loop": has_append_in_loop(tree),
        "has_add_in_loop": has_add_in_loop(tree),
        "has_membership_or_index_in_loop": has_membership_or_index_in_loop(tree),
        "max_nesting_depth": max_nesting_depth(tree),
        "same_level_loop_count": count_same_level_loops(tree),
    }

print('Module 1 ready.')

Module 1 ready.


In [32]:
def detect_candidate_patterns(features):
    candidates = []
    if features.get("has_append_in_loop"):
        candidates.append("P4_Loop_to_Comprehension")
    if features.get("has_add_in_loop"):
        candidates.append("P5_Set_Membership_Optimization")
    if features.get("has_membership_or_index_in_loop"):
        candidates.append("P6_Dictionary_Lookup_Optimization")
    if features.get("max_nesting_depth", 0) >= 2:
        candidates.append("P1_Nested_Loop_Elimination")
    if features.get("same_level_loop_count", 0) > 1:
        candidates.append("P2_Loop_Reduction")
    if not candidates:
        candidates.append("P3_Internal_Optimization")
    return candidates

PATTERN_ID_MAP = {
    "P1_Nested_Loop_Elimination": "P1", "P2_Loop_Reduction": "P2",
    "P3_Internal_Optimization": "P3", "P4_Loop_to_Comprehension": "P4",
    "P5_Set_Membership_Optimization": "P5", "P6_Dictionary_Lookup_Optimization": "P6",
}

def retrieve_poi(final_pattern_key, kb):
    pattern_id = PATTERN_ID_MAP.get(final_pattern_key)
    if pattern_id is None or pattern_id not in kb:
        return {}, {}
    entry = kb[pattern_id]
    return entry.get("poi", {}), entry

print('Module 2 ready.')

Module 2 ready.


In [ ]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
import time

GPT_MODEL = "gpt-5.4-mini"  # verify exact model string in OpenAI's docs before running

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

def call_gen_llm(prompt, max_new_tokens=1024, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.15,
                max_completion_tokens=max_new_tokens,
            )
            content = response.choices[0].message.content
            return content.strip() if content else ''
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

print(f'{GPT_MODEL} ready (temperature=0.3, max_new_tokens=1024).')

In [34]:
def build_repair_prompt(pattern_name, poi, slow_code):
    activation_str = "\n".join(f"- {c}" for c in poi.get("activation_conditions", poi.get("when", [])))
    constraints_str = "\n".join(f"- {c}" for c in poi.get("constraints", []))
    return f"""You are an expert Python engineer specializing in performance optimization.

Apply the following optimization pattern to the slow code below.

Pattern: {pattern_name}
Trigger conditions:
{activation_str}
Transformation: {poi.get('transformation', poi.get('how', ''))}
Constraints:
{constraints_str}

Slow code:
{slow_code}

Return ONLY the optimized Python code. No markdown. No explanations."""

def extract_code_from_response(response):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

print('Module 5 ready.')

Module 5 ready.


In [35]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

print('Module 6 ready.')

Module 6 ready.


In [36]:
def build_feedback_prompt(pattern_name, poi, slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')
    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {})
        feedback = (
            f"Your previous attempt failed a test case.\n"
            f"Input:\n{case.get('stdin', '')}\n"
            f"Expected output:\n{case.get('expected', '')}\n"
            f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\n\n"
            f"Before rewriting, first explain in a comment WHY your previous logic "
            f"produced the wrong output for this specific input, then fix it. "
            f"Common causes: off-by-one errors in loop bounds, incorrect edge-case "
            f"handling (empty input, single element), or changed variable semantics "
            f"during optimization (e.g., using a set removed needed duplicates)."
        )
    elif reason == 'insufficient_speedup':
        feedback = (
            f"Your previous attempt was correct but NOT fast enough "
            f"({validation_result.get('speedup')}x, need >= {SPEEDUP_THRESHOLD}x). "
            f"A speedup this low usually means only micro-optimizations were made. "
            f"Consider: reducing algorithmic complexity (O(n^2) -> O(n) or O(n log n)), "
            f"using built-in functions (sum(), sorted(), collections.Counter) instead "
            f"of manual loops, or eliminating redundant recomputation inside loops."
        )
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_repair_prompt(pattern_name, poi, slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback on that attempt:
{feedback}

Generate a NEW, corrected version that addresses this feedback."""

print('Improved feedback prompt builder ready.')

Improved feedback prompt builder ready.


In [37]:
def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['problem_id'])
                except Exception:
                    continue
    return completed

def repair_sample(sample, kb, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    problem_id = sample.get('problem_id')
    slow_code = sample['input']
    features = extract_structural_features(slow_code)
    if features is None:
        return {'problem_id': problem_id, 'final_verdict': 'PARSE_ERROR'}
    candidates = detect_candidate_patterns(features)
    final_pattern = candidates[0]
    poi, kb_entry = retrieve_poi(final_pattern, kb)
    prompt = build_repair_prompt(final_pattern, poi, slow_code)
    current_code = extract_code_from_response(call_gen_llm(prompt))
    history = []
    for attempt in range(1, max_attempts + 1):
        result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        result['attempt'] = attempt
        history.append(result)
        if result['verdict'] == 'ACCEPT':
            return {'problem_id': problem_id, 'final_verdict': 'ACCEPT', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code}
        if attempt == max_attempts:
            return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'detected_pattern': final_pattern,
                    'candidates': candidates, 'num_attempts': attempt, 'history': history, 'final_code': current_code}
        fb_prompt = build_feedback_prompt(final_pattern, poi, slow_code, current_code, result)
        current_code = extract_code_from_response(call_gen_llm(fb_prompt))
    return {'problem_id': problem_id, 'final_verdict': 'RETRY_EXHAUSTED', 'history': history}

print('repair_sample ready.')

repair_sample ready.


In [38]:
def run_batch(samples, kb, test_cases_dir, output_path, max_attempts=MAX_ATTEMPTS):
    completed = load_completed_problem_ids(output_path)
    print(f'Already completed: {len(completed)} / {len(samples)}')
    with open(output_path, 'a') as out_f:
        for i, sample in enumerate(samples):
            problem_id = sample.get('problem_id')
            if problem_id in completed:
                continue
            print(f'[{i+1}/{len(samples)}] {problem_id}...')
            start = time.time()
            try:
                record = repair_sample(sample, kb, test_cases_dir, max_attempts)
                record['elapsed_seconds'] = round(time.time() - start, 2)
            except Exception as e:
                record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}
            out_f.write(json.dumps(record) + '\n')
            out_f.flush()
    print('Batch run complete.')

print('run_batch ready.')

run_batch ready.


In [ ]:
run_batch(test_samples, pattern_knowledge_base, TEST_CASES_DIR, OUTPUT_PATH, max_attempts=MAX_ATTEMPTS)

In [41]:
results = []
with open(OUTPUT_PATH) as f:
    for line in f:
        results.append(json.loads(line))

def score_at_k(results, k):
    total = len(results)
    opt_count = 0
    pass1_count = 0
    for r in results:
        history = r.get('history', [])[:k]
        if not history:
            continue
        if any(h.get('verdict') == 'ACCEPT' for h in history):
            opt_count += 1
        if history[0].get('verdict') == 'ACCEPT' or history[0].get('reason') == 'insufficient_speedup':
            pass1_count += 1
    print(f'k={k}: %OPT = {100*opt_count/total:.1f}%, Pass@1 = {100*pass1_count/total:.1f}%')

for k in range(1, 11):
    score_at_k(results, k)

k=1: %OPT = 23.4%, Pass@1 = 81.8%
k=2: %OPT = 38.0%, Pass@1 = 81.8%
k=3: %OPT = 43.8%, Pass@1 = 81.8%
k=4: %OPT = 48.4%, Pass@1 = 81.8%
k=5: %OPT = 49.5%, Pass@1 = 81.8%
k=6: %OPT = 51.6%, Pass@1 = 81.8%
k=7: %OPT = 53.1%, Pass@1 = 81.8%
k=8: %OPT = 53.1%, Pass@1 = 81.8%
k=9: %OPT = 53.1%, Pass@1 = 81.8%
k=10: %OPT = 54.7%, Pass@1 = 81.8%


In [ ]:
print(results[0])

In [45]:
import base64
from IPython.display import HTML

with open(OUTPUT_PATH, 'rb') as f:
    file_data = f.read()

b64 = base64.b64encode(file_data).decode()
filename = OUTPUT_PATH.split('/')[-1]

html = f'''
<a download="{filename}" href="data:application/json;base64,{b64}" 
   style="padding:10px 20px; background:#0066cc; color:white; text-decoration:none; border-radius:5px;">
   ⬇ Download {filename}
</a>
'''
HTML(html)

In [46]:
import statistics

def score_at_k_full(results, k):
    total = len(results)
    opt_count = 0
    pass1_count = 0
    speedups = []

    for r in results:
        history = r.get('history', [])[:k]
        if not history:
            continue
        accepted_in_k = [h for h in history if h.get('verdict') == 'ACCEPT']
        if accepted_in_k:
            opt_count += 1
            speedups.append(accepted_in_k[0]['speedup'])
        if history[0].get('verdict') == 'ACCEPT' or history[0].get('reason') == 'insufficient_speedup':
            pass1_count += 1

    median_sp = statistics.median(speedups) if speedups else None
    mean_sp = statistics.mean(speedups) if speedups else None

    line = f'k={k}: %OPT = {100*opt_count/total:.1f}%, Pass@1 = {100*pass1_count/total:.1f}%'
    if median_sp:
        line += f', Median speedup = {median_sp:.3f}x, Mean speedup = {mean_sp:.3f}x'
    print(line)

for k in range(1, 11):
    score_at_k_full(results, k)

k=1: %OPT = 23.4%, Pass@1 = 81.8%, Median speedup = 1.799x, Mean speedup = 1.797x
k=2: %OPT = 38.0%, Pass@1 = 81.8%, Median speedup = 1.784x, Mean speedup = 1.797x
k=3: %OPT = 43.8%, Pass@1 = 81.8%, Median speedup = 1.790x, Mean speedup = 1.854x
k=4: %OPT = 48.4%, Pass@1 = 81.8%, Median speedup = 1.784x, Mean speedup = 1.836x
k=5: %OPT = 49.5%, Pass@1 = 81.8%, Median speedup = 1.784x, Mean speedup = 1.834x
k=6: %OPT = 51.6%, Pass@1 = 81.8%, Median speedup = 1.693x, Mean speedup = 1.806x
k=7: %OPT = 53.1%, Pass@1 = 81.8%, Median speedup = 1.692x, Mean speedup = 1.824x
k=8: %OPT = 53.1%, Pass@1 = 81.8%, Median speedup = 1.692x, Mean speedup = 1.824x
k=9: %OPT = 53.1%, Pass@1 = 81.8%, Median speedup = 1.692x, Mean speedup = 1.824x
k=10: %OPT = 54.7%, Pass@1 = 81.8%, Median speedup = 1.692x, Mean speedup = 1.812x
